# 02 — Warehouse Validation

Verification of the staging and analytics layers after `build_warehouse.py`:
row counts, dimension integrity, and a star-schema join test.

In [1]:
import duckdb
con = duckdb.connect("../aml.duckdb", read_only=True)

con.sql("""
    SELECT
        is_fraud,
        COUNT(*)                            AS n,
        AVG(orig_emptied::INT)              AS pct_emptied,
        AVG(balance_inconsistent_orig::INT) AS pct_inconsistent
    FROM stg_transactions
    GROUP BY is_fraud
""").show()

┌──────────┬─────────┬─────────────────────┬───────────────────────┐
│ is_fraud │    n    │     pct_emptied     │   pct_inconsistent    │
│ boolean  │  int64  │       double        │        double         │
├──────────┼─────────┼─────────────────────┼───────────────────────┤
│ true     │    8213 │  0.9755266041641301 │ 0.0024351637647631805 │
│ false    │ 6354407 │ 0.23803464272905403 │    0.2501286115289751 │
└──────────┴─────────┴─────────────────────┴───────────────────────┘



In [2]:
con.sql("""
    SELECT
        transaction_type,
        COUNT(*)                                    AS n,
        AVG(balance_inconsistent_orig::INT)         AS pct_inconsistent,
        AVG(orig_balance_missing::INT)              AS pct_balance_missing
    FROM stg_transactions
    GROUP BY transaction_type
    ORDER BY n DESC
""").show()

┌──────────────────┬─────────┬─────────────────────┬─────────────────────┐
│ transaction_type │    n    │  pct_inconsistent   │ pct_balance_missing │
│     varchar      │  int64  │       double        │       double        │
├──────────────────┼─────────┼─────────────────────┼─────────────────────┤
│ CASH_OUT         │ 2237500 │ 0.43150256983240226 │ 0.45845765363128493 │
│ PAYMENT          │ 2151495 │ 0.18203574723622412 │  0.3598637226672616 │
│ CASH_IN          │ 1399284 │                 0.0 │                 0.0 │
│ TRANSFER         │  532909 │ 0.42408553805621596 │  0.5306403157011798 │
│ DEBIT            │   41432 │ 0.15215292527514965 │ 0.14862907897277466 │
└──────────────────┴─────────┴─────────────────────┴─────────────────────┘



### Data quality: balance consistency after correction

An initial check (old_balance − amount = new_balance) flagged **79.9%** of
legitimate transactions as inconsistent — an implausible result. Two causes
were identified and corrected: CASH_IN uses the opposite formula
(old + amount = new), and zero balances in PaySim encode missing data rather
than a true zero. After excluding both, the rate falls to **25.0%**, and
CASH_IN from 100% to 0%.


#### Two observations

1. **A residual 15-43% inconsistency remains across outgoing transaction types.**

   This reflects PaySim's synthetic generation rather than a processing error —
   the simulator does not maintain strict balance continuity.

2. **Fraudulent transactions are ~100x more balance-consistent than legitimate ones.**

   (0.24% vs 25.0%). This is an artefact of how PaySim generates fraud, not a
   property of real fraud. This feature should be treated with caution in
   modelling, as it may leak simulator structure rather than genuine signal.

In [13]:
con.sql("SHOW TABLES").show()

┌──────────────────────┐
│         name         │
│       varchar        │
├──────────────────────┤
│ dim_account          │
│ dim_time             │
│ dim_transaction_type │
│ fact_transactions    │
│ raw_transactions     │
│ stg_transactions     │
└──────────────────────┘



In [4]:
con.sql("""
    SELECT 'fact_transactions' AS table_name, COUNT(*) AS n FROM fact_transactions
    UNION ALL SELECT 'dim_time', COUNT(*) FROM dim_time
    UNION ALL SELECT 'dim_account', COUNT(*) FROM dim_account
    UNION ALL SELECT 'dim_transaction_type', COUNT(*) FROM dim_transaction_type
""").show()

┌──────────────────────┬─────────┐
│      table_name      │    n    │
│       varchar        │  int64  │
├──────────────────────┼─────────┤
│ fact_transactions    │ 6362620 │
│ dim_time             │     743 │
│ dim_account          │ 9073900 │
│ dim_transaction_type │       5 │
└──────────────────────┴─────────┘



In [5]:
con.sql("""
    SELECT s AS missing_step
    FROM range(1, 745) AS t(s)
    LEFT JOIN dim_time d ON t.s = d.step
    WHERE d.step IS NULL
""").show()

┌──────────────┐
│ missing_step │
│    int64     │
├──────────────┤
│          744 │
└──────────────┘



In [6]:
con.sql("SELECT MIN(step), MAX(step) FROM stg_transactions").show()

┌───────────┬───────────┐
│ min(step) │ max(step) │
│   int32   │   int32   │
├───────────┼───────────┤
│         1 │       743 │
└───────────┴───────────┘



### dim_time completeness

The simulation spans steps 1–743, not the full 744. `dim_time` contains all
743 steps present in the data — the "missing" step 744 simply has no
transactions, so the dimension is complete relative to its source.

In [7]:
con.sql("""
    SELECT account_type, COUNT(*) AS n
    FROM dim_account
    GROUP BY account_type
""").show()

┌──────────────┬─────────┐
│ account_type │    n    │
│   varchar    │  int64  │
├──────────────┼─────────┤
│ customer     │ 6923499 │
│ merchant     │ 2150401 │
└──────────────┴─────────┘



In [8]:
con.sql("""
    SELECT
        n_received,
        COUNT(*) AS n_merchants
    FROM dim_account
    WHERE account_type = 'merchant'
    GROUP BY n_received
    ORDER BY n_received
    LIMIT 10
""").show()

┌────────────┬─────────────┐
│ n_received │ n_merchants │
│   int64    │    int64    │
├────────────┼─────────────┤
│          1 │     2149308 │
│          2 │        1092 │
│          3 │           1 │
└────────────┴─────────────┘



### Dataset structure: merchants are effectively single-use

99.95% of merchant accounts (2,149,308 of 2,150,401) appear in exactly one
transaction. Merchants in PaySim are generated per-payment rather than
representing recurring businesses.

Consequences:
- Merchant-history-based detection rules are not viable (no history exists)
- Merchant-side account features carry no information for the model
- 24% of dim_account rows are single-use identifiers

This is a structural property of the simulator, not a data quality issue.

In [9]:
con.sql("""
    SELECT
        t.hour_of_day,
        tt.description,
        COUNT(*)             AS n,
        SUM(f.is_fraud::INT) AS n_fraud
    FROM fact_transactions f
    JOIN dim_time t              ON f.step = t.step
    JOIN dim_transaction_type tt ON f.transaction_type = tt.transaction_type
    WHERE t.is_night
    GROUP BY 1, 2
    ORDER BY n_fraud DESC
    LIMIT 10
""").show()

┌─────────────┬─────────────────────────────────────────────┬───────┬─────────┐
│ hour_of_day │                 description                 │   n   │ n_fraud │
│    int32    │                   varchar                   │ int64 │ int128  │
├─────────────┼─────────────────────────────────────────────┼───────┼─────────┤
│           1 │ Customer withdraws cash via a merchant      │  1256 │     186 │
│           1 │ Funds sent to another mobile money customer │   666 │     186 │
│           4 │ Customer withdraws cash via a merchant      │   373 │     183 │
│           4 │ Funds sent to another mobile money customer │   259 │     183 │
│           0 │ Customer withdraws cash via a merchant      │  3898 │     180 │
│           5 │ Customer withdraws cash via a merchant      │   517 │     179 │
│           5 │ Funds sent to another mobile money customer │   387 │     179 │
│           0 │ Funds sent to another mobile money customer │  2107 │     178 │
│           2 │ Customer withdraws cash 

### Finding: strict transfer-then-cashout pairing

Within each night hour, TRANSFER and CASH_OUT contain the *same number* of
fraudulent transactions — hour 1: both 186; hour 4: both 183; hour 5: both
179. Each fraudulent transfer is matched by exactly one cash-out in the same
hour, a textbook layering-then-extraction pattern.

### Finding: night-time concentration

Legitimate activity collapses at night while fraud continues steadily. At
hour 4, fraud accounts for 70.7% of TRANSFER volume (183 of 259), against a
baseline fraud rate of 0.13%.

Implications: `is_night` is a strong candidate feature, and paired
TRANSFER→CASH_OUT detection is the highest-priority rule for Phase 3.